<a href="https://colab.research.google.com/github/Tar-ive/dl_basics/blob/main/Implement_RNN_PyTorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
#Creating Tensors
V_data = [1., 2., 3.]
V = torch.tensor(V_data)
V

tensor([1., 2., 3.])

In [4]:
# Creating a matrix
M_data = [[1., 2., 3.], [4., 5. ,6]]
M = torch.tensor(M_data)
M

tensor([[1., 2., 3.],
        [4., 5., 6.]])

In [6]:
# Creating a 3D Tensor of size 1x4x2
T_data = [[[1.,2.], [3., 4.], [5., 6.], [7., 8.]]]
T = torch.tensor(T_data)
T.shape

torch.Size([1, 4, 2])

In [7]:
# Creating a 3D Tensor of size 2x2x2
T_data = [[[1.,2.], [3., 4.]],
          [[5., 6.], [7., 8.]]]
T = torch.tensor(T_data)
T.shape

torch.Size([2, 2, 2])

In [10]:
#Index into V and get a scalar (0 dimensional tensor)
V[0].item()

1.0

In [13]:
#Index into M and get a matrix
M[0]


tensor([1., 2., 3.])

In [41]:
T[1]

tensor([[5., 6.],
        [7., 8.]])

In [47]:
# Operating on tensors
x = torch.tensor([1., 2., 3.])
y = torch.tensor([4., 5. ,6])
x+y

tensor([5., 7., 9.])

In [48]:
#Concatination
# By default concatinations happen on first axis (concatenates rows)
x_1= torch.randn(2,5)
y_1= torch.randn(3,5)
z_1 = torch.cat([x_1, y_1])
z_1

tensor([[ 0.2829,  0.9491,  0.2978, -0.3136,  0.3761],
        [ 1.2182,  0.3171, -1.7147, -1.3483, -0.6239],
        [-0.5470,  0.6975,  1.8200,  1.3399, -1.3636],
        [ 0.7594, -1.4210, -0.3226, -0.7091,  0.0920],
        [ 0.7467, -0.4411, -0.0657, -0.0731, -0.7277]])

In [53]:
#Concatenate cols
x_2= torch.randn(2,3)
y_2= torch.randn(2,5)
#second arg specifies which axis to concat along
z_2 = torch.cat([x_2, y_2],1)
z_2

tensor([[-1.0579,  0.0943,  1.3392, -0.4597, -0.6828,  0.5665, -0.1065,  0.4508],
        [ 0.6764, -1.0860,  0.9727,  1.0433,  0.5845, -0.0078,  0.0335, -1.1307]])

In [54]:
#requires_grad = True -> tensor object keep track of how it was created
x = torch.tensor([1.,2.,3], requires_grad = True)

In [55]:
x

tensor([1., 2., 3.], requires_grad=True)

In [58]:
y = torch.tensor([1.,2.,3], requires_grad = True)
z=x+y
z.grad_fn

In [59]:
s = z.sum()
print(s)
s.grad_fn

tensor(12., grad_fn=<SumBackward0>)


$$\frac{\partial s}{\partial x_0}$$

$$s = \overbrace{x_0 + y_0}^\text{$z_0$} + \overbrace{x_1 + y_1}^\text{$z_1$} + \overbrace{x_2 + y_2}^\text{$z_2$}$$

In [61]:
s.backward()
print(x.grad)

tensor([2., 2., 2.])


In [63]:
x=torch.randn(2,2)
y = torch.randn(2,2)

In [65]:
# we created tensors, by default with requires_grad=False

In [68]:
x.requires_grad , y.requires_grad

(False, False)

In [71]:
z= x+y
print(z.grad_fn)

None


In [74]:
#.requires_grad cahanges an existing tensors requires_grad in place.
x= x.requires_grad_()
y= y.requires_grad_()
z=x+y
z.requires_grad

True

In [77]:
#just taking value of z and detaching its computation history
new_z = z.detach()

In [79]:
print(new_z.grad_fn)

None


In [81]:
x.requires_grad

True

In [82]:
(x**2).requires_grad

True

In [84]:
with torch.no_grad():
  print((x**2).requires_grad)

False


# RNN code

In [2]:
import torch
import torch.optim as optim
import torch.nn as nn


In [86]:
# Create data
torch.manual_seed(42)
sequence_length = 10
num_samples = 100

#Generate dataset
y = torch.sin(torch.linspace(0, 4*3.14159, steps = num_samples).unsqueeze(1))

# Preparing data for RNN - in out sequence
def create_in_out_sequences(data, sequence_length):
  in_seq, out_seq = [], []
  for i in range(len(data) - sequence_length):
    in_seq.append(data[i:i+sequence_length])
    out_seq.append(data[i + sequence_length])
  return torch.stack(in_seq), torch.stack(out_seq)

X_seq, y_seq = create_in_out_sequences(y, sequence_length)


In [90]:
# Define RNN Model
class RNNModel(nn.Module):
  def __init__(self, input_dim=1, hidden_dim = 50, output_dim = 1):
    super().__init__()
    self.hidden_dim = hidden_dim

    # weight matrix for input and hidden state
    self.W_ih = nn.Parameter(torch.randn(input_dim, hidden_dim)*0.1)
    self.W_hh = nn.Parameter(torch.randn(hidden_dim, hidden_dim)*0.1)
    self.b_h = nn.Parameter(torch.zeros(hidden_dim))

    # Output layer
    self.output_layer = nn.Linear(hidden_dim, output_dim)

    # Activation
    self.tanh = nn.Tanh()

  def forward(self,x):
    batch_size, sequence_length, _ = x.size()
    h_t = torch.zeros(batch_size, self.hidden_dim, device = x.device)

    for t in range(sequence_length):
      x_t = x[:,t,:]
      h_t = self.tanh(x_t @ self.W_ih + h_t @ self.W_hh + self.b_h)
    output = self.output_layer(h_t)
    return output

In [91]:
# Initialize the model, loss function, and optimizer
model = RNNModel()
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training loop
epochs = 500
for epoch in range(epochs):
    for sequences, labels in zip(X_seq, y_seq):
        sequences = sequences.unsqueeze(0)  # Add batch dimension
        labels = labels.unsqueeze(0)  # Add batch dimension

        # Forward pass
        outputs = model(sequences)
        loss = criterion(outputs, labels)

        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(f"Epoch [{epoch + 1}/{epochs}], Loss: {loss.item():.4f}")

Epoch [1/500], Loss: 0.0721
Epoch [2/500], Loss: 0.0034
Epoch [3/500], Loss: 0.0746
Epoch [4/500], Loss: 0.0022
Epoch [5/500], Loss: 0.0521
Epoch [6/500], Loss: 0.0001
Epoch [7/500], Loss: 0.0048
Epoch [8/500], Loss: 0.0160
Epoch [9/500], Loss: 0.0002
Epoch [10/500], Loss: 0.0002
Epoch [11/500], Loss: 0.0005
Epoch [12/500], Loss: 0.0004
Epoch [13/500], Loss: 0.0000
Epoch [14/500], Loss: 0.0000
Epoch [15/500], Loss: 0.0000
Epoch [16/500], Loss: 0.0001
Epoch [17/500], Loss: 0.0005
Epoch [18/500], Loss: 0.0084
Epoch [19/500], Loss: 0.0000
Epoch [20/500], Loss: 0.0011
Epoch [21/500], Loss: 0.0002
Epoch [22/500], Loss: 0.0001
Epoch [23/500], Loss: 0.0000
Epoch [24/500], Loss: 0.0000
Epoch [25/500], Loss: 0.0000
Epoch [26/500], Loss: 0.0000
Epoch [27/500], Loss: 0.0000
Epoch [28/500], Loss: 0.0000
Epoch [29/500], Loss: 0.0000
Epoch [30/500], Loss: 0.0000
Epoch [31/500], Loss: 0.0000
Epoch [32/500], Loss: 0.0000
Epoch [33/500], Loss: 0.0001
Epoch [34/500], Loss: 0.0001
Epoch [35/500], Loss: 0

In [92]:
#Testing on new data
X_test =torch.sin(torch.linspace(4*3.14159, 8*3.14159, steps = num_samples).unsqueeze(1))

#Reshape to (batch_size, sequence_length, input_size)
X_test = X_test.unsqueeze(0)

with torch.no_grad():
    predictions = model(X_test) # Predict the next value of the sine wave.
    print(f"Preceding three values: {X_test[:, -3:, :].tolist()}")
    print(f"Predictions for new sequence: {predictions.tolist()}")


Preceding three values: [[[-0.25116774439811707], [-0.12661167979240417], [-2.0281453544157557e-05]]]
Predictions for new sequence: [[0.12763744592666626]]
